In [ ]:
import torch
import torch.nn as nn
import librosa
import gradio as gr
import numpy as np
import soundfile as sf
import os
import base64
import random
from transformers import AutoProcessor, AutoModelForCTC, Wav2Vec2Processor, Wav2Vec2ForCTC
from Levenshtein import ratio
import re
import difflib

from feature_extraction_ko import extract_accuracy_features, extract_fluency_features, extract_prosody_features
from PronunciationModel_ko import PronunciationModel  
from asr_visualization_for_realtime_gradio_ko import visualize_asr_analysis  # ASR 정렬 시각화 모듈 추가
from phoneme_feedback_ko import analyze_phoneme_errors  # 피드백 모듈 불러오기


# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# phoneme 모델 불러오기
processor = AutoProcessor.from_pretrained("slplab/wav2vec2-XLS-R-300m_KoreanPhonene_ASR_spoken_by_foreigners")
model_phoneme = AutoModelForCTC.from_pretrained("slplab/wav2vec2-XLS-R-300m_KoreanPhonene_ASR_spoken_by_foreigners").to(device)

# ASR 모델 불러오기
model_name = "slplab/wav2vec2-xls-r_Korean_ASR_by_foreigners"
asr_processor = Wav2Vec2Processor.from_pretrained(model_name)
asr_model = Wav2Vec2ForCTC.from_pretrained(model_name).to(device)

# 여러 문장 및 정답 음소 설정 (추가 문장을 원하는 만큼 넣을 수 있음)
sentences = [
    {
        "sentence": "얼굴 보니 너 무슨 안 좋은 일 있구나",
        "phoneme": "ㅓ ㄹ ㄱ ㅜ ㄹ ㅂ ㅗ ㄴ ㅣ ㄴ ㅓ ㅁ ㅜ ㅅ ㅡ ㄴ ㅏ ㄴ ㅈ ㅗ ㅡ ㄴ ㅣ ㄹ ㅣ ㄷ ㄲ ㅜ ㄴ ㅏ"
    },
    {
        "sentence": "뒷일은 나중에 생각하고 급한 불부터 끕시다",
        "phoneme": "ㄷ ㅟ ㄴ ㄴ ㅣ ㄹ ㅡ ㄴ ㄴ ㅏ ㅈ ㅜ ㅇ ㅔ ㅅ ㅐ ㅇ ㄱ ㅏ ㅋ ㅏ ㄱ ㅗ ㄱ ㅡ ㅍ ㅏ ㄴ ㅂ ㅜ ㄹ ㅂ ㅜ ㅌ ㅓ ㄲ ㅡ ㅂ ㅆ ㅣ ㄷ ㅏ"
    },
    {
        "sentence": "주말에 별일 없으면 우리 집에 와서 밥 같이 먹어요",
        "phoneme": "ㅈ ㅜ ㅁ ㅏ ㄹ ㅔ ㅂ ㅕ ㄹ ㄹ ㅣ ㄹ ㅓ ㅂ ㅆ ㅡ ㅁ ㅕ ㄴ ㅜ ㄹ ㅣ ㅈ ㅣ ㅂ ㅔ ㅘ ㅅ ㅓ ㅂ ㅏ ㅂ ㄱ ㅏ ㅊ ㅣ ㅁ ㅓ ㄱ ㅓ ㅛ"
    },
    {
        "sentence": "저는 지금 음료수하고 꽃을 사러 갔다 와야 돼요",
        "phoneme": "ㅈ ㅓ ㄴ ㅡ ㄴ ㅈ ㅣ ㄱ ㅡ ㅁ ㅡ ㅁ ㄴ ㅛ ㅅ ㅜ ㅎ ㅏ ㄱ ㅗ ㄲ ㅗ ㅊ ㅡ ㄹ ㅅ ㅏ ㄹ ㅓ ㄱ ㅏ ㄷ ㄸ ㅏ ㅘ ㅑ ㄷ ㅙ ㅛ"
    },
    {
        "sentence": "내일 늦으시면 안 됩니다",
        "phoneme": "ㄴ ㅐ ㅣ ㄹ ㄴ ㅡ ㅈ ㅡ ㅅ ㅣ ㅁ ㅕ ㄴ ㅏ ㄴ ㄷ ㅚ ㅁ ㄴ ㅣ ㄷ ㅏ"
    },
    {
        "sentence": "내가 먹던 라면이 어디로 갔지?",
        "phoneme": "ㄴ ㅐ ㄱ ㅏ ㅁ ㅓ ㄱ ㄸ ㅓ ㄴ ㄹ ㅏ ㅁ ㅕ ㄴ ㅣ ㅓ ㄷ ㅣ ㄹ ㅗ ㄱ ㅏ ㄷ ㅉ ㅣ"
    },
    {
        "sentence": "불만이 있으면 직접 가서 말해 봐",
        "phoneme": "ㅂ ㅜ ㄹ ㅁ ㅏ ㄴ ㅣ ㅣ ㅆ ㅡ ㅁ ㅕ ㄴ ㅈ ㅣ ㄱ ㅉ ㅓ ㅂ ㄱ ㅏ ㅅ ㅓ ㅁ ㅏ ㄹ ㅎ ㅐ ㅂ ㅘ"
    },
    {
        "sentence": "여기 비빔밥 한 그릇 갖다 주세요",
        "phoneme": "ㅕ ㄱ ㅣ ㅂ ㅣ ㅂ ㅣ ㅁ ㅃ ㅏ ㅂ ㅎ ㅏ ㄴ ㄱ ㅡ ㄹ ㅡ ㄷ ㄱ ㅏ ㄷ ㄸ ㅏ ㅈ ㅜ ㅅ ㅔ ㅛ"
    },
    {
        "sentence": "권력과 공권력의 차이점이 뭔지 아세요",
        "phoneme": "ㄱ ㅝ ㄹ ㄹ ㅕ ㄱ ㄲ ㅘ ㄱ ㅗ ㅇ ㄲ ㅝ ㄴ ㄴ ㅕ ㄱ ㅔ ㅊ ㅏ ㅣ ㅉ ㅓ ㅁ ㅣ ㅁ ㅝ ㄴ ㅈ ㅣ ㅏ ㅅ ㅔ ㅛ"
    },
    {
        "sentence": "이 빵 달콤한 게 진짜 맛있다",
        "phoneme": "ㅣ ㅃ ㅏ ㅇ ㄷ ㅏ ㄹ ㅋ ㅗ ㅁ ㅎ ㅏ ㄴ ㄱ ㅔ ㅈ ㅣ ㄴ ㅉ ㅏ ㅁ ㅏ ㅅ ㅣ ㄷ ㄸ ㅏ"
    }
]

# 랜덤으로 하나의 문장을 선택
selected_item = random.choice(sentences)
default_sentence = selected_item["sentence"]
default_phoneme = selected_item["phoneme"]

In [ ]:
def remove_punctuation(text):
    return re.sub(r"[^\w\sㄱ-ㅎㅏ-ㅣ가-힣]", "", text)


def analyze_word_difference(ref_sentence, hyp_sentence):
    """
    Compare reference vs. hypothesis at the word level,
    and return a feedback string listing missing or replaced words.
    """
    # 1) Remove punctuation & convert to lowercase
    ref_clean = remove_punctuation(ref_sentence.lower())
    hyp_clean = remove_punctuation(hyp_sentence.lower())
    
    # 2) Split into word tokens
    ref_words = ref_clean.split()
    hyp_words = hyp_clean.split()

    matcher = difflib.SequenceMatcher(None, ref_words, hyp_words)
    opcodes = matcher.get_opcodes()

    missing_words = []
    replaced_words = []
    inserted_words = []

    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'delete':
            # ref_words[i1:i2] were deleted (missing in the hypothesis)
            missing_words.extend(ref_words[i1:i2])
        elif tag == 'replace':
            # ref_words[i1:i2] replaced by hyp_words[j1:j2]
            replaced_words.append((ref_words[i1:i2], hyp_words[j1:j2]))
        elif tag == 'insert':
            # hyp_words[j1:j2] are inserted words in the hypothesis
            inserted_words.extend(hyp_words[j1:j2])
        # 'equal' means those words match

    feedback = []
    if missing_words:
        feedback.append(f"발화하지 않은 단어: {', '.join(missing_words)}")

    if replaced_words:
        feedback.append("잘못 발음된 단어:")
        for refs, hyps in replaced_words:
            ref_str = ' '.join(refs)
            hyp_str = ' '.join(hyps)
            feedback.append(f" - {ref_str} → {hyp_str}")

    if inserted_words:
        feedback.append(f"추가하여 발화한 단어: {', '.join(inserted_words)}")

    # 모든 리스트가 비어 있으면 = 원문과 완전히 일치한 경우
    if not missing_words and not replaced_words and not inserted_words:
        feedback.append("모든 단어를 언급하였습니다.")

    # 잘못 발음된 단어 개수(= 교체된 원본 단어 총 개수) 계산
    replaced_word_count = sum(len(ref_list) for ref_list, _ in replaced_words)


    return "\n".join(feedback), replaced_word_count



In [ ]:

# 예측 함수
def predict_scores_with_asr(audio_path, correct_phoneme, correct_sentence):
    #  **16kHz가 아닌 경우에만 변환**
    y, sr = librosa.load(audio_path, sr=None)  # 원본 SR 유지
    if sr != 16000:
        y = librosa.resample(y, orig_sr=sr, target_sr=16000)  # 16kHz 변환
        sf.write(audio_path, y, 16000)  # 변환된 오디오를 다시 저장


    # ASR 예측
    audio, _ = librosa.load(audio_path, sr=16000)
    inputs = asr_processor(audio, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = asr_model(inputs.input_values).logits
    emissions = torch.log_softmax(logits, dim=-1)
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = asr_processor.batch_decode(predicted_ids)[0]
    transcription = transcription.replace("<unk>", "").strip()

    # 음소 예측
    phoneme_inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        phoneme_logits = model_phoneme(phoneme_inputs.input_values).logits
    predicted_phoneme_ids = torch.argmax(phoneme_logits, dim=-1)
    predicted_phonemes = processor.batch_decode(predicted_phoneme_ids.cpu())[0]
    
    # --------------------------
    # 정답 문장과 전사 결과 간의 유사도 확인
    print(transcription)
    clean_ref = remove_punctuation(correct_sentence.lower())
    clean_hyp = remove_punctuation(transcription.lower())
    sentence_similarity = ratio(clean_ref, clean_hyp)
    # sentence_similarity = ratio(correct_sentence.lower(), transcription.lower())
    # phoneme_similarity = ratio(correct_phoneme.lower(), predicted_phonemes.lower())
    
    # 평가 기준 설정
    sentence_threshold = 0.5  # 문장 유사도 기준 (기존 0.7 → 0.6으로 낮춤)
    # phoneme_threshold = 0.5  # 음소 유사도 기준 (문장이 틀려도 발음이 비슷하면 평가 가능)
    
    # 평가 여부 결정
    if sentence_similarity < sentence_threshold:
        # 문장 유사도와 음소 유사도가 모두 낮으면 평가 진행 X
        return None
    # --------------------------
    
    
    # 정해진 문장과 유사도가 충분하다면 나머지 평가 진행

    # 평가 모델
    model = PronunciationModel(
                               len(extract_fluency_features(audio_path)),
                               len(extract_prosody_features(audio_path))
    ).to(device)

    state_dict = torch.load("best_model.pth")
    model.load_state_dict({k.replace("module.", ""): v for k, v in state_dict.items()})
    model.eval()

    # 평가 요소 관련 특성 추출
    accuracy_features = torch.tensor(extract_accuracy_features(audio_path, correct_phoneme, processor, model_phoneme, device), dtype=torch.float32).unsqueeze(0).to(device)
    fluency_features = torch.tensor(extract_fluency_features(audio_path), dtype=torch.float32).unsqueeze(0).to(device)
    prosody_features = torch.tensor(extract_prosody_features(audio_path), dtype=torch.float32).unsqueeze(0).to(device)

    # 평가 모델 예측 
    with torch.no_grad():
        fluency_score, prosody_score = model(fluency_features, prosody_features)
    
    fluency_score = round(fluency_score.item() / 0.5) * 0.5
    prosody_score = round(prosody_score.item() / 0.5) * 0.5

    # 최대 점수를 5.0으로 제한
    fluency_score = max(0.0, min(fluency_score, 5.0))
    prosody_score = max(0.0, min(prosody_score, 5.0))

    # Accuracy Score 직접 계산 (정답 음소 vs 예측 음소)
    print(accuracy_features)
    accuracy_score = accuracy_features.mean().item() * 5  # 0~5점 범위로 변환
    accuracy_score = round(accuracy_score / 0.5) * 0.5  # 0.5 단위로 반올림
    
    # Completeness Score 계산
    alpha = 0.7
    phoneme_similarity = ratio(correct_phoneme.lower(), predicted_phonemes.lower())
    combined_similarity = ( alpha * sentence_similarity + (1 - alpha) * phoneme_similarity)
    completeness_score = round(combined_similarity * 5, 1)
    completeness_score = round(completeness_score / 0.5) * 0.5

    # 수정된 analyze_word_difference 호출 → (피드백, 잘못 발음된 단어 개수)
    word_diff_feedback, replaced_word_count = analyze_word_difference(correct_sentence, transcription)
    # 잘못 발음된 단어 1개당 0.3점씩 감점
    completeness_score -= replaced_word_count * 0.3
    # 0점 미만이면 0, 5점 초과면 5로 보정
    completeness_score = max(0.0, min(completeness_score, 5.0))

    # Final Score 계산
    scores = [accuracy_score, fluency_score, prosody_score, completeness_score]
    final_score = round(0.4 * min(scores) + 0.2 * sum(sorted(scores)[1:]), 2)

    # 피드백 생성
    phoneme_feedback = analyze_phoneme_errors(correct_phoneme.lower(), predicted_phonemes.lower(), correct_sentence)
    phoneme_feedback = "\n".join(phoneme_feedback)  # 리스트를 문자열로 변환

    # ASR 정렬 및 시각화 실행
    fig, word_segments_audio = visualize_asr_analysis(audio_path, transcription, emissions)
    
    return accuracy_score, fluency_score, prosody_score, completeness_score, final_score, transcription, predicted_phonemes, phoneme_feedback, word_diff_feedback, fig, word_segments_audio




In [ ]:

def get_base64_audio(file_path):
    with open(file_path, "rb") as f:
        audio_bytes = f.read()
    base64_audio = base64.b64encode(audio_bytes).decode("utf-8")
    return f"data:audio/wav;base64,{base64_audio}"


# 새로운 문장을 선택하는 함수
def change_sentence():
    global default_sentence, default_phoneme
    selected_item = random.choice(sentences)
    default_sentence = selected_item["sentence"]
    default_phoneme = selected_item["phoneme"]
    return default_sentence

# Gradio 인터페이스
def evaluate_pronunciation(audio):
    if audio is None:
        return "Error: No audio received. Please check your microphone or re-upload the audio.", None, None
    print(f"Received audio path: {audio}")  # 디버깅용 로그 출력

    if not os.path.exists(audio) or os.path.getsize(audio) == 0:
        return "Error: No valid audio file received.", None, None

    # 파일 경로인 경우 바로 처리
    if isinstance(audio, str):  
        temp_audio_path = audio  # 이미 파일 경로이므로 그대로 사용
    else:
        temp_audio_path = "temp_audio.wav"

        if isinstance(audio, tuple):  # (sr, data) 형태로 오는 경우
            sr, audio = audio
        else:
            sr = 16000  # 기본 16kHz 가정

        if len(audio.shape) > 1:  # 다중 채널 오디오인 경우, 첫 번째 채널만 사용
            audio = audio[0, :]
            
        if not np.issubdtype(audio.dtype, np.floating):
            audio = audio.astype(np.float32) / np.iinfo(audio.dtype).max

        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000, res_type="kaiser_fast")

        sf.write(temp_audio_path, audio, 16000)

    # 평가 함수 호출
    result = predict_scores_with_asr(temp_audio_path, default_phoneme, default_sentence)
    
    # result가 튜플이 아니라 (None, transcription)이면, 정해진 문장이 아닌 경우임
    if result is None or result[0] is None:
        return "주어진 문장을 발화해주세요", None, None
    
    # 정상인 경우 unpack
    accuracy_score, fluency_score, prosody_score, completeness_score, final_score, transcription, predicted_phonemes, phoneme_feedback, word_diff_feedback, fig, word_segments_audio = result
    
    result_text = f"""
    Pronunciation Evaluation Results
    - Accuracy Score: {accuracy_score}
    - Fluency Score: {fluency_score}
    - Prosody Score: {prosody_score}
    - Completeness Score: {completeness_score}
    - Final Score: {final_score}

    Transcription Results
    - Given Sentence: {default_sentence}
    - Pronounced Sentence: {transcription}

    - Target Phonemes: {default_phoneme}
    - Predicted Phonemes: {predicted_phonemes}

    Phoneme Feedback:
    {phoneme_feedback}

    Word-Level Feedback:
    {word_diff_feedback}
    """

    # Create an HTML snippet for word segments
    word_audio_html = "<div style='display: flex; flex-wrap: wrap;'>"
    
    for label, score, file_path in word_segments_audio:
        abs_path = os.path.abspath(file_path)  # 절대 경로 변환
    
        if os.path.exists(abs_path) and os.path.getsize(abs_path) > 0:
            audio_src = get_base64_audio(abs_path)
            word_audio_html += f"""
            <div style="margin: 10px; text-align: center;">
                <strong>{label} ({score:.2f})</strong><br>
                <audio controls>
                    <source src="{audio_src}" type="audio/wav">
                    Your browser does not support the audio element.
                </audio>
            </div>
            """
        else:
            print(f"Skipping {label}: Missing or empty file {abs_path}")
    
    word_audio_html += "</div>"
    
    return result_text, fig, word_audio_html


In [ ]:
with gr.Blocks(css="""
.output-class {height: 800px !important; width: 1200px !important;}
.image-class {height: 600px !important; width: 1100px !important;} 
""") as interface:

    gr.Markdown("### Pronunciation Assessment with ASR Visualization")

    # 문장 표시
    sentence_display = gr.Markdown(f"**Please read the following sentence aloud:**\n\n**{default_sentence}**")

    with gr.Row():
        audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", interactive=True)

    submit_button = gr.Button("Evaluate")

    # "Change Sentence" 버튼 추가
    change_button = gr.Button("Change Sentence")
    
    # 버튼 클릭 시 새로운 문장으로 변경
    change_button.click(fn=change_sentence, inputs=[], outputs=[sentence_display])

    
    with gr.Row():
        output_text = gr.Textbox(label="Evaluation Results", interactive=False)
        
    with gr.Row():
        output_image = gr.Image(label="Alignment Visualization", elem_id="image-class", type="filepath")

    with gr.Row():
        output_word_audio = gr.HTML(label="Word-Level Audio Segments")


    submit_button.click(fn=evaluate_pronunciation, inputs=audio_input, outputs=[output_text, output_image, output_word_audio])

# Gradio 실행
interface.launch(server_name="127.0.0.1", server_port=7861, share=True)
